# 04 Channel Routing and Session Boundaries (OpenClaw, 2026)

## What This Lesson Is
Model multi-channel agent behavior with explicit session boundaries to prevent context bleed.

## Scientific Lens
- Concept: Session isolation by channel + sender + agent intent.
- Measure: Isolation integrity rate across synthetic mixed-channel workloads.
- Validity Limit: Isolation keys are only as reliable as upstream metadata quality.


## How It Works
1. Build composite session identity key from channel/sender/intent.
2. Validate no cross-session collisions in deterministic simulation.
3. Run live call using channel-derived user key to emulate routed execution.


In [ ]:
import os
from openai import OpenAI

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_client()
    resp = client.chat.completions.create(
        model="openclaw",
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
events = [
    {"channel":"telegram","sender":"alice","intent":"review"},
    {"channel":"slack","sender":"alice","intent":"review"},
    {"channel":"telegram","sender":"bob","intent":"ops"},
]
keys = [f"{e['channel']}::{e['sender']}::{e['intent']}" for e in events]
assert len(set(keys)) == len(events)


In [ ]:
# Live Demo
try:
    channel = "telegram"
    sender = "alice"
    session_user = f"{channel}:{sender}:review"
    print(ask_openclaw("Summarize the current review context in 2 bullets.", user=session_user))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add trust level to session keys and route low-trust senders to a constrained agent id.
2. Simulate 1,000 events and compute key-collision rate.
3. Add a per-channel retention policy and verify key schema supports enforcement.

## Validation Checklist
- Session key contract is explicit and collision-tested.
- Channel routing is modeled as AI-system behavior, not UI-only logic.
- Live demo uses routed session identity through OpenClaw API.

## Further Reading
- OpenClaw pairing/channels docs directory: https://docs.openclaw.ai/start/docs-directory
- OpenClaw API session behavior: https://docs.openclaw.ai/gateway/openai-http-api
